In [ ]:
import pandas as pd
from blox2 import load_features
from sklearn.preprocessing import StandardScaler

features_df = load_features("../data/zinc_dft/feature_list_mfp2048_rdd.npz")
values_df = pd.read_csv("../data/zinc_dft/properties.csv")

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd

@dataclass
class EvalResult:
    metric: str
    mean: float
    std: float
    per_try: np.ndarray
    n_tries: int
    n_train: int
    n_test: int
    d_obj: int

def evaluate_predictor(predictor, features: pd.DataFrame, properties: pd.DataFrame, n_samples: int, n_tries: int, metric: str="rmse", seed:int=0, verbose: bool=True) -> EvalResult:
    """
    Args:
        predictor: instance of Predictor subclass (must implement fit, and pred or pred_samples).
        features: DataFrame of shape (N, d_feat) for all data.
        properties: DataFrame of shape (N, d_obj) for all data.
        n_samples: Number of training samples per try.
        n_tries: Number of repeated trials.
        metric: "rmse", "mae", or "r2".

    Returns:
        EvalResult: summary + per-try scores.
    """
    if len(features) != len(properties):
        raise ValueError(f"features and properties must have same number of rows. "
                         f"Got {len(features)=}, {len(properties)=}.")

    N = len(features)
    if not (1 <= n_samples < N):
        raise ValueError(f"n_samples must be in [1, N-1]. Got {n_samples=} with {N=}.")

    if n_tries <= 0:
        raise ValueError(f"n_tries must be positive. Got {n_tries=}.")

    metric = metric.lower().strip()
    if metric not in {"rmse", "mae", "r2"}:
        raise ValueError(f"Unsupported metric: {metric}. Choose from rmse/mae/r2.")

    rng = np.random.default_rng(seed)

    X_all = features.to_numpy()
    Y_all = properties.to_numpy()
    scaler = StandardScaler()
    Y_all = scaler.fit_transform(Y_all)

    d_obj = Y_all.shape[1]
    scores = np.empty(n_tries, dtype=float)

    # precompute indices once
    all_idx = np.arange(N)

    def _predict(p, X: np.ndarray) -> np.ndarray:
        # Prefer pred(); fallback to pred_samples() mean.
        if hasattr(p, "pred"):
            y = p.pred(X)
            if y is not None:
                y = np.asarray(y)
                if y.ndim != 2:
                    raise ValueError(f"predictor.pred must return 2D array (m, d_obj). Got {y.shape=}.")
                return y

        if not hasattr(p, "pred_samples"):
            raise ValueError("Predictor must implement pred() or pred_samples().")

        y_s = p.pred_samples(X)
        y_s = np.asarray(y_s)
        if y_s.ndim != 3:
            raise ValueError(f"predictor.pred_samples must return 3D array (m, n_samples, d_obj). Got {y_s.shape=}.")
        return y_s.mean(axis=1)

    def _score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        if y_true.shape != y_pred.shape:
            raise ValueError(f"Shape mismatch: {y_true.shape=} vs {y_pred.shape=}.")
        err = y_pred - y_true

        if metric == "rmse":
            # average over all dims and samples
            return float(np.sqrt(np.mean(err ** 2)))
        if metric == "mae":
            return float(np.mean(np.abs(err)))
        # r2 (global, flatten)
        y_true_f = y_true.reshape(-1)
        y_pred_f = y_pred.reshape(-1)
        ss_res = float(np.sum((y_true_f - y_pred_f) ** 2))
        ss_tot = float(np.sum((y_true_f - np.mean(y_true_f)) ** 2))
        return float(1.0 - (ss_res / ss_tot if ss_tot > 0 else np.nan))

    for t in range(n_tries):
        train_idx = rng.choice(all_idx, size=n_samples, replace=False)
        test_mask = np.ones(N, dtype=bool)
        test_mask[train_idx] = False
        test_idx = all_idx[test_mask]

        X_train = X_all[train_idx]
        Y_train = Y_all[train_idx]
        X_test = X_all[test_idx]
        Y_test = Y_all[test_idx]

        predictor.fit(X_train, Y_train)
        Y_pred = _predict(predictor, X_test)

        scores[t] = _score(Y_test, Y_pred)

    mean = float(np.nanmean(scores))
    std = float(np.nanstd(scores, ddof=1)) if n_tries >= 2 else 0.0

    if verbose:
        print(
            f"[evaluate_predictor_holdout] metric={metric} "
            f"mean={mean:.6g} std={std:.6g} "
            f"(n_tries={n_tries}, n_train={n_samples}, n_test={N - n_samples}, d_obj={d_obj})"
        )

    return EvalResult(metric=metric, mean=mean, std=std, per_try=scores, n_tries=n_tries, n_train=n_samples, n_test=N - n_samples, d_obj=d_obj)

In [9]:
from blox2.predictor import LightGBMPredictor

predictor = LightGBMPredictor(n_estimators=50, learning_rate=0.05, num_leaves=31)
evaluate_predictor(predictor, features_df, values_df, n_samples=100, n_tries=20)

[evaluate_predictor_holdout] metric=rmse mean=34.9998 std=0.872167 (n_tries=20, n_train=100, n_test=87528, d_obj=2)


EvalResult(metric='rmse', mean=34.999776503008846, std=0.8721665671565124, per_try=array([34.33420521, 34.02995276, 34.66280447, 34.74058737, 35.53052734,
       34.34399629, 36.34244303, 34.36680915, 35.10058144, 35.3720641 ,
       35.20023121, 34.89091038, 37.22344228, 34.25013354, 35.35284636,
       36.40801663, 35.3210766 , 34.08060942, 34.20436294, 34.23992953]), n_tries=20, n_train=100, n_test=87528, d_obj=2)